In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

storage      = "stairqualitydev01"
bronze_path  = f"abfss://bronze@{storage}.dfs.core.windows.net/openaq/*/*/*/*/*.json"
target_table = "airquality.silver.pm25_hourly"

In [0]:
raw = (
    spark.read
         .option("multiLine", True)
         .json(bronze_path)
         .withColumn("source_file", F.col("_metadata.file_path"))
)

print("Files read:", raw.count())
raw.printSchema()

In [0]:
exploded = raw.select(
    F.regexp_extract("source_file", r"openaq/([^/]+)/", 1).alias("city"),
    F.regexp_extract("source_file", r"_(\d+)_\d{8}_\d{8}\.json$", 1).cast("long").alias("sensor_id"),
    "source_file",
    F.explode("results").alias("r"),
)

display(exploded.limit(5))

In [0]:
typed = exploded.select(
    "city",
    "sensor_id",
    F.to_timestamp("r.period.datetimeFrom.utc").alias("time_utc"),
    F.col("r.value").cast("double").alias("pm25_ugm3"),
    F.col("r.parameter.units").alias("units"),
    F.col("r.coverage.percentCoverage").cast("double").alias("coverage_pct"),
    F.col("r.flagInfo.hasFlags").alias("has_flags"),
    "source_file",
)

# Count bad values before removing them
bad = typed.filter(F.col("pm25_ugm3").isNull() | (F.col("pm25_ugm3") < 0))
print("Invalid values removed:", bad.count())

silver = (
    typed.filter(F.col("pm25_ugm3").isNotNull() & (F.col("pm25_ugm3") >= 0))
         .dropDuplicates(["sensor_id", "time_utc"])
         .withColumn("processed_at", F.current_timestamp())
)

display(silver.limit(10))

In [0]:
expected_hours = 50 * 24   # 1 Aug → 19 Sep = 50 days

display(
    silver.groupBy("city", "sensor_id").agg(
        F.count("*").alias("hours_with_data"),
        F.min("time_utc").alias("first_hour"),
        F.max("time_utc").alias("last_hour"),
        F.round(F.avg("pm25_ugm3"), 1).alias("avg_pm25"),
        F.round(F.count("*") / expected_hours * 100, 1).alias("completeness_pct"),
    )
)

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists(target_table):

    target = DeltaTable.forName(spark, target_table)

    (
        target.alias("t")
        .merge(
            silver.alias("s"),
            "t.sensor_id = s.sensor_id AND t.time_utc = s.time_utc"
        )
        .whenMatchedUpdateAll(condition="t.pm25_ugm3 <> s.pm25_ugm3")
        .whenNotMatchedInsertAll()
        .execute()
    )
else:

    silver.write.format("delta").saveAsTable(target_table)
